# Last Human on Earth - Colab render
T4 GPU render of the 60 s vertical short (1080x1920, h264). Run cells top to bottom.

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'], capture_output=True, text=True).stdout or 'NO GPU')
print('node', subprocess.run(['node','-v'], capture_output=True, text=True).stdout.strip())
print(sys.version)

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 libcairo2 xvfb libgl1-mesa-dri libegl1 libglu1-mesa mesa-utils ffmpeg > /dev/null
print('deps ok')

In [ ]:
import os
if not os.path.isdir('/content/last-human-project'):
    !git clone https://github.com/shahrukhkathat199/last-human-project.git
%cd /content/last-human-project
!npm ci --legacy-peer-deps 2>&1 | tail -3
!npx remotion browser ensure

### 1. Probe which WebGL backend actually works on this runtime

In [ ]:
import subprocess, os, shutil
os.chdir('/content/last-human-project')
print('xvfb-run:', shutil.which('xvfb-run'))

def attempt(mode, use_xvfb):
    out = 'out/probe_' + mode + '_' + str(int(use_xvfb)) + '.mp4'
    cmd = ['npx','remotion','render','src/index.ts','LastHumanShort',out,
           '--frames=0-2','--concurrency=1','--gl=' + mode,'--timeout=120000']
    if use_xvfb:
        cmd = ['xvfb-run','-a','-s','-screen 0 1280x1920x24'] + cmd
    r = subprocess.run(cmd, capture_output=True, text=True)
    log = r.stdout + r.stderr
    ok = r.returncode == 0 and os.path.exists(out)
    bad = [l for l in log.splitlines() if ('WebGL' in l or 'Could not create' in l)]
    print('MODE', mode, 'xvfb', use_xvfb, '=>', 'OK' if ok else 'FAIL', (bad[0][:140] if bad else ''))
    return ok

working = None
for mode, x in [('swangle',False),('egl',False),('egl',True),('angle',True),('swangle',True),('angle',False)]:
    if attempt(mode, x):
        working = (mode, int(x))
        break

if working:
    open('/content/glmode.txt','w').write(working[0] + ' ' + str(working[1]))
    print('WORKING:', working)
else:
    print('NO WORKING GL MODE')

### 2. Full render

In [ ]:
import subprocess, os
os.chdir('/content/last-human-project')
mode, xvfb = open('/content/glmode.txt').read().split()
cmd = ['npx','remotion','render','src/index.ts','LastHumanShort','out/last-human.mp4',
       '--codec=h264','--crf=20','--scale=1','--concurrency=2',
       '--gl=' + mode,'--timeout=300000']
if xvfb == '1':
    cmd = ['xvfb-run','-a','-s','-screen 0 1280x1920x24'] + cmd
print('RUN', ' '.join(cmd), flush=True)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
tail = []
for line in p.stdout:
    tail.append(line.rstrip())
    if len(tail) > 30: tail.pop(0)
    if ('Rendered ' in line) or ('Encoded ' in line) or ('Error' in line):
        print(line.rstrip(), flush=True)
p.wait()
print('EXIT', p.returncode)
print('\n'.join(tail[-20:]))

### 3. Verify and download

In [ ]:
import os, subprocess
f = '/content/last-human-project/out/last-human.mp4'
print(round(os.path.getsize(f)/1e6, 1), 'MB')
print(subprocess.run(['ffprobe','-v','error','-show_entries','stream=codec_type,codec_name,width,height,duration','-of','default=noprint_wrappers=1',f], capture_output=True, text=True).stdout)
from google.colab import files
files.download(f)